# Phase 2: Collect Activations & Train SAEs

Load the shared NAR trained in Phase 1, collect processor activations for each
algorithm, and train a BatchTopK SAE per algorithm. Also collects concept labels
for downstream analysis.

In [ ]:
# --- Environment setup (Colab + local) ---
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", ""))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = "/content/nar-mechinterp"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aniervs/nar-mechinterp.git", REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    os.chdir(REPO_DIR)

    # Install only the deps Colab doesn't ship (torch, numpy, matplotlib, etc. are preinstalled)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "salsa-clrs @ git+https://github.com/jkminder/SALSA-CLRS.git",
                    "torch-geometric", "dm-haiku", "einops", "hydra-core", "omegaconf",
                    "rich", "scikit-learn", "plotly", "seaborn"], check=True)

    # Add repo root to Python path so our packages are importable
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Patch salsa-clrs to enable extra algorithms
    subprocess.run([sys.executable, "scripts/patch_salsaclrs.py"], check=True)
    print(f"Working directory: {os.getcwd()}")
    print("Colab setup complete.")
else:
    if os.path.basename(os.getcwd()) == "experiments":
        os.chdir("..")
    print(f"Running locally from {os.getcwd()}")

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from data import (
    AVAILABLE_ALGORITHMS,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
)
from models import NARModel
from interp import (
    BatchTopKSAE, SAETrainer, SAEOutput,
    ActivationCollector, make_activation_dataloader,
    collect_concept_labels,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

## Configuration

In [ ]:
ALGORITHMS = AVAILABLE_ALGORITHMS
print(f"Algorithms: {ALGORITHMS}")

LOCAL_DEBUG = False

if LOCAL_DEBUG:
    HIDDEN_DIM = 32
    NUM_LAYERS = 2
    NUM_HEADS = 4
    SAE_STEPS = 5_000
    SAE_NUM_SAMPLES = 500
else:
    HIDDEN_DIM = 128
    NUM_LAYERS = 4
    NUM_HEADS = 8
    SAE_STEPS = 50_000
    SAE_NUM_SAMPLES = 10_000

PROCESSOR_TYPE = "mpnn"
NUM_NODES = 16
EDGE_PROB = 0.2

EXPANSION_FACTOR = 4
SAE_K = 16 if LOCAL_DEBUG else 32
SAE_LR = 3e-4
SAE_BATCH_SIZE = 256

# Paths — use Google Drive on Colab for persistence
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/nar-mechinterp")
    CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "multi_algo"
    RESULTS_ROOT = DRIVE_ROOT / "results" / "multi_algo"
else:
    CHECKPOINT_DIR = Path("checkpoints") / "multi_algo"
    RESULTS_ROOT = Path("results") / "multi_algo"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'LOCAL DEBUG' if LOCAL_DEBUG else 'FULL SCALE'}")
print(f"SAE: expansion={EXPANSION_FACTOR}x, k={SAE_K}, steps={SAE_STEPS}")
print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Results: {RESULTS_ROOT}")

## Load Trained NAR

In [ ]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

ckpt = torch.load(CHECKPOINT_DIR / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded NAR checkpoint (epoch {ckpt['epoch']}, avg_acc={ckpt.get('avg_acc', 'N/A')})")
if 'per_algo_acc' in ckpt:
    for algo, acc in sorted(ckpt['per_algo_acc'].items()):
        print(f"  {algo:25s} acc={acc:.3f}")

## Collect Activations & Concept Labels Per Algorithm

For each algorithm:
1. Create a fresh dataloader (with a fixed seed for reproducibility)
2. Run the NAR in eval mode and collect processor hidden states
3. Extract ground-truth concept labels from algorithm hints
4. Save both to disk

In [ ]:
for algo in ALGORITHMS:
    algo_dir = RESULTS_ROOT / algo
    algo_dir.mkdir(parents=True, exist_ok=True)
    act_path = algo_dir / "activations.pt"
    lbl_path = algo_dir / "concept_labels.pt"

    # --- Activations ---
    if act_path.exists():
        print(f"[{algo}] Activations already saved, skipping collection.")
    else:
        print(f"[{algo}] Collecting activations...")
        spec = get_algorithm_spec(algo)
        output_types, _ = spec_to_model_types(spec)
        act_loader = get_clrs_dataloader(
            algo, "train",
            batch_size=32, num_samples=SAE_NUM_SAMPLES,
            num_nodes=NUM_NODES, edge_probability=EDGE_PROB,
            seed=SEED + 100, shuffle=False,
        )
        collector = ActivationCollector(model, spec=spec, device=device)
        activations = collector.collect(act_loader, output_types=output_types)
        torch.save(activations, act_path)
        print(f"  Saved: {activations.shape} -> {act_path}")
        del activations
        torch.cuda.empty_cache()

    # --- Concept labels ---
    if lbl_path.exists():
        print(f"[{algo}] Concept labels already saved, skipping.")
    else:
        print(f"[{algo}] Collecting concept labels...")
        spec = get_algorithm_spec(algo)
        label_loader = get_clrs_dataloader(
            algo, "train",
            batch_size=32, num_samples=SAE_NUM_SAMPLES,
            num_nodes=NUM_NODES, edge_probability=EDGE_PROB,
            seed=SEED + 100, shuffle=False,
        )
        concept_result = collect_concept_labels(label_loader, spec, algo)
        torch.save({
            "labels": concept_result.labels,
            "descriptions": concept_result.concept_descriptions,
            "algorithm": concept_result.algorithm,
        }, lbl_path)
        print(f"  Concepts: {list(concept_result.labels.keys())}")
        del concept_result
    print()

## Train SAE Per Algorithm

Train a BatchTopK SAE on each algorithm's activations.

In [ ]:
dict_size = HIDDEN_DIM * EXPANSION_FACTOR

for algo in ALGORITHMS:
    algo_dir = RESULTS_ROOT / algo
    sae_path = algo_dir / "sae.pt"

    if sae_path.exists():
        print(f"[{algo}] SAE already trained, skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"Training SAE for: {algo}")
    print(f"{'='*60}")

    activations = torch.load(algo_dir / "activations.pt")
    print(f"  Activations: {activations.shape}")

    sae = BatchTopKSAE(input_dim=HIDDEN_DIM, dict_size=dict_size, k=SAE_K).to(device)
    trainer = SAETrainer(sae, lr=SAE_LR, resample_dead_every=0)
    loader = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

    sae_losses = []
    step = 0
    while step < SAE_STEPS:
        for (batch,) in loader:
            if step >= SAE_STEPS:
                break
            batch = batch.to(device)
            output = trainer.train_step(batch)
            if step % 500 == 0:
                sae_losses.append({
                    'step': step,
                    'loss': output.loss.item(),
                    'recon': output.reconstruction_loss.item(),
                    'l0': output.l0.item(),
                })
            if step % 5_000 == 0:
                print(f"  [{algo}] step {step}/{SAE_STEPS} | "
                      f"loss={output.loss.item():.4f} | L0={output.l0.item():.1f}")
            step += 1

    # Save SAE
    torch.save({
        "state_dict": sae.state_dict(),
        "config": sae.get_config(),
        "algorithm": algo,
    }, sae_path)

    # Save training losses
    torch.save(sae_losses, algo_dir / "sae_losses.pt")
    print(f"  Saved SAE -> {sae_path}")

    del sae, trainer, activations, loader
    torch.cuda.empty_cache()

print("\nAll SAEs trained.")

## Collect Per-Layer Activations (Optional)

Collect activations from each processor layer separately, for per-layer
analysis in Phase 3.

In [ ]:
for algo in ALGORITHMS:
    algo_dir = RESULTS_ROOT / algo
    pla_path = algo_dir / "per_layer_activations.pt"

    if pla_path.exists():
        print(f"[{algo}] Per-layer activations already saved, skipping.")
        continue

    print(f"[{algo}] Collecting per-layer activations...")
    spec = get_algorithm_spec(algo)
    output_types, _ = spec_to_model_types(spec)

    act_loader = get_clrs_dataloader(
        algo, "train",
        batch_size=32, num_samples=SAE_NUM_SAMPLES,
        num_nodes=NUM_NODES, edge_probability=EDGE_PROB,
        seed=SEED + 100, shuffle=False,
    )

    model.eval()
    layer_act_lists = {i: [] for i in range(NUM_LAYERS)}

    with torch.no_grad():
        for batch in act_loader:
            inputs, _, _ = batch_to_model_inputs(batch, spec, device)
            num_steps = batch.lengths.max().item()
            output = model(
                inputs=inputs,
                output_types=output_types,
                num_steps=num_steps,
                return_activations=True,
            )
            acts = output.activations
            for layer_idx in range(NUM_LAYERS):
                for step_dict in acts['layer_activations'][layer_idx]:
                    h = step_dict['node_mlp_output']
                    layer_act_lists[layer_idx].append(h.reshape(-1, h.shape[-1]).cpu())

    per_layer_acts = {i: torch.cat(layer_act_lists[i], dim=0) for i in range(NUM_LAYERS)}
    for i in range(NUM_LAYERS):
        print(f"  Layer {i}: {per_layer_acts[i].shape}")
    torch.save(per_layer_acts, pla_path)
    print(f"  Saved -> {pla_path}")
    del per_layer_acts, layer_act_lists
    torch.cuda.empty_cache()

print("\nDone.")

## SAE Training Curves

In [ ]:
n_algos = len(ALGORITHMS)
n_cols = min(5, n_algos)
n_rows = (n_algos + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = np.array(axes).flatten()

for i, algo in enumerate(ALGORITHMS):
    loss_path = RESULTS_ROOT / algo / "sae_losses.pt"
    if not loss_path.exists():
        axes[i].set_title(f"{algo} (no data)")
        continue

    sae_losses = torch.load(loss_path, weights_only=True)
    steps = [d['step'] for d in sae_losses]
    axes[i].plot(steps, [d['loss'] for d in sae_losses], linewidth=0.8)
    axes[i].set_title(algo, fontsize=10)
    axes[i].set_xlabel("Step")
    if i % n_cols == 0:
        axes[i].set_ylabel("Loss")

for i in range(n_algos, len(axes)):
    axes[i].set_visible(False)

fig.suptitle("SAE Training Loss Per Algorithm", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

Activations, concept labels, and trained SAEs are saved per algorithm under
`RESULTS_ROOT/<algorithm>/`. Use `03_analyze.ipynb` for feature interpretation
and cross-algorithm comparison.